# Final 20-event D-Wave Leap hybrid BQM campaign

This notebook submits the **unchanged validated 945-binary / 419,814-edge QUBO** for each of the 20 frozen CAISO events.

Default resource request: `20 events × 30 s = 600 s` of requested hybrid runtime.

**Important:** run preflight first. No CPLEX or simulated annealing is rerun here.

In [6]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd

from ai_spot_curtailment_dvfs_qubo_core import (
    DEFAULT_CONFIG, build_qubo, decode_sample, build_dimod_bqm, qubo_energy
)
from ai_spot_curtailment_dwave_hybrid import (
    create_sampler, sampler_identity, min_time_limit_seconds,
    solve_one_event, build_campaign_fingerprint,
    initialize_or_validate_campaign, event_is_complete,
    save_event_result, rebuild_summary,
)

DATA_DIR = Path('prepared_ai_curtailment_dvfs')
BASELINE_DIR = Path('baseline_results')
OUT = Path('dwave_hybrid_results')
CORE_PY = Path('ai_spot_curtailment_dvfs_qubo_core.py')

TIME_LIMIT = 30.0
EVENT_IDS = list(range(20))
RESUME = True
PREFLIGHT_ONLY = False   # FIRST RUN: keep True. Change to False only after preflight passes.
LABEL_PREFIX = 'AI-curtailment-DVFS'

print(f'Planned D-Wave submissions: {len(EVENT_IDS)}')
print(f'Requested hybrid runtime: {len(EVENT_IDS)*TIME_LIMIT/60:.2f} min')

Planned D-Wave submissions: 20
Requested hybrid runtime: 10.00 min


In [7]:
cfg = DEFAULT_CONFIG
portfolio = pd.read_csv(DATA_DIR/'benchmark_spot_jobs_dvfs.csv')
cal = pd.read_csv(DATA_DIR/'emerald_dvfs_calibration.csv')
profiles = pd.read_csv(DATA_DIR/'event_profiles_long.csv', parse_dates=['timestamp'])
manifest = json.loads((DATA_DIR/'revised_data_and_qubo_manifest.json').read_text())

# Strict frozen-model checks.
assert int(manifest['qubo_variables']) == 945
assert int(manifest['qubo_edges']) == 419814
assert int(manifest['selected_jobs']) == 18
assert int(manifest['selected_portfolio_total_gpus']) == 256
assert abs(float(manifest['lambda_once']) - 20.0) < 1e-12
assert len(EVENT_IDS) == 20 and EVENT_IDS == list(range(20))

q0 = build_qubo(portfolio, cal, profiles[profiles.event_id == 0], cfg)
assert q0['n_variables'] == 945
assert q0['n_edges'] == 419814
print(f'Frozen QUBO: {q0["n_variables"]} binaries, {q0["n_edges"]:,} edges, density={100*q0["density"]:.2f}%')
print(f'Analytical same-job penalty bound: {q0["lambda_sufficient_bound"]:.6g} < lambda={cfg.lambda_once:g}')

Frozen QUBO: 945 binaries, 419,814 edges, density=94.12%
Analytical same-job penalty bound: 15.36 < lambda=20


In [8]:
# Authenticate and inspect the actual solver. This does not submit an optimization.
sampler = create_sampler()
sinfo = sampler_identity(sampler)
print(json.dumps(sinfo, indent=2))

bqm0 = build_dimod_bqm(q0)
min_t0 = min_time_limit_seconds(sampler, bqm0)
print(f'Event-0 BQM minimum accepted time: {min_t0:.6g} s')
assert not np.isfinite(min_t0) or TIME_LIMIT >= min_t0, '30-s request is below the solver minimum.'

fingerprint = build_campaign_fingerprint(DATA_DIR, CORE_PY, TIME_LIMIT, EVENT_IDS)
initialize_or_validate_campaign(OUT, fingerprint, sinfo, resume=RESUME)

if PREFLIGHT_ONLY:
    print('\nPREFLIGHT PASSED. No D-Wave optimization was submitted.')
    print('Set PREFLIGHT_ONLY=False and rerun the notebook to start the 20-event campaign.')

{
  "python_sampler_class": "LeapHybridSampler",
  "imported_hybrid_class": "LeapHybridBQMSampler",
  "solver_name": "hybrid_binary_quadratic_model_version2p",
  "solver_id": "hybrid_binary_quadratic_model_version2p",
  "quota_conversion_rate": 1,
  "maximum_number_of_variables": 1000000,
  "maximum_number_of_biases": 200000000,
  "minimum_time_limit_property": [
    [
      1,
      3.0
    ],
    [
      1024,
      3.0
    ],
    [
      4096,
      10.0
    ],
    [
      10000,
      40.0
    ],
    [
      30000,
      200.0
    ],
    [
      100000,
      600.0
    ],
    [
      1000000,
      600.0
    ]
  ],
  "version": "2.4"
}
Event-0 BQM minimum accepted time: 3 s


In [9]:
if not PREFLIGHT_ONLY:
    campaign_start = time.perf_counter()
    for n, eid in enumerate(EVENT_IDS, start=1):
        if RESUME and event_is_complete(OUT, eid):
            print(f'[{n:02d}/{len(EVENT_IDS)}] event {eid:02d}: already complete -> skipped')
            continue

        ep = profiles[profiles.event_id == eid].sort_values('slot').copy()
        q = build_qubo(portfolio, cal, ep, cfg)
        assert q['n_variables'] == q0['n_variables']
        assert q['n_edges'] == q0['n_edges']

        print('\n' + '='*94)
        print(f'[{n:02d}/{len(EVENT_IDS)}] EVENT {eid:02d} | submit {q["n_variables"]}-binary BQM | time_limit={TIME_LIMIT:g}s')
        sample, meta = solve_one_event(q, sampler=sampler, time_limit=TIME_LIMIT, label_prefix=LABEL_PREFIX)
        decoded = decode_sample(sample, portfolio, ep, q, cfg)
        m = decoded['metrics']

        # Exact returned-sample validation.
        assert abs(meta['solver_energy_reported'] - meta['qubo_energy_recomputed']) < 1e-5
        physical_feasible = (
            m['duplicate_starts'] == 0
            and m['max_envelope_violation_kW'] <= 1e-9
            and m['max_concurrent_gpus'] <= cfg.field_cluster_gpus
        )
        meta['physical_feasible'] = bool(physical_feasible)
        meta['campaign_sequence'] = int(n)

        save_event_result(OUT, eid, sample, decoded, meta, ep)
        summary = rebuild_summary(OUT)

        print(f"objective={m['objective_direct']:.8f} | RMSE={m['tracking_RMSE_kW']:.4f} kW | "
              f"jobs={m['n_jobs_scheduled']} | duplicate={m['duplicate_starts']} | "
              f"envelope_violation={m['max_envelope_violation_kW']:.3g} kW | feasible={physical_feasible}")
        print(f"client wall={meta['client_wall_time_s']:.2f}s | run={meta['run_time_s']:.3f}s | "
              f"charge={meta['charge_time_s']:.3f}s | qpu={meta['qpu_access_time_s']:.3f}s")
        print(f'Checkpoint saved: {OUT}/events/event_{eid:02d}/')

    elapsed = time.perf_counter() - campaign_start
    summary = rebuild_summary(OUT)
    print('\n' + '='*94)
    print(f'Campaign notebook wall time this invocation: {elapsed/60:.2f} min')
    print(f'Completed events on disk: {len(summary)}/{len(EVENT_IDS)}')
    display(summary)


[01/20] EVENT 00 | submit 945-binary BQM | time_limit=30s
objective=0.01309066 | RMSE=0.5494 kW | jobs=10 | duplicate=0 | envelope_violation=0 kW | feasible=True
client wall=37.86s | run=29.994s | charge=29.994s | qpu=1.602s
Checkpoint saved: dwave_hybrid_results/events/event_00/

[02/20] EVENT 01 | submit 945-binary BQM | time_limit=30s
objective=0.01441664 | RMSE=0.5766 kW | jobs=9 | duplicate=0 | envelope_violation=0 kW | feasible=True
client wall=39.28s | run=29.990s | charge=29.990s | qpu=1.653s
Checkpoint saved: dwave_hybrid_results/events/event_01/

[03/20] EVENT 02 | submit 945-binary BQM | time_limit=30s
objective=0.01068337 | RMSE=0.4963 kW | jobs=9 | duplicate=0 | envelope_violation=0 kW | feasible=True
client wall=39.84s | run=29.987s | charge=29.987s | qpu=1.602s
Checkpoint saved: dwave_hybrid_results/events/event_02/

[04/20] EVENT 03 | submit 945-binary BQM | time_limit=30s
objective=0.01004088 | RMSE=0.4812 kW | jobs=7 | duplicate=0 | envelope_violation=0 kW | feasible

,objective_direct,tracking_objective,once_penalty,duplicate_starts,n_jobs_scheduled,tracking_RMSE_kW,max_envelope_violation_kW,envelope_violation_kWh,max_concurrent_gpus,cluster_gpu_capacity,...,qubo_energy_recomputed,energy_abs_difference,charge_time_raw,charge_time_s,run_time_raw,run_time_s,qpu_access_time_raw,qpu_access_time_s,physical_feasible,campaign_sequence
0,0.013091,0.013091,0.0,0,10,0.549407,0.000000,0.000000,61,256,...,0.013091,2.470246e-15,29994200,29.994200,29994200,29.994200,1601916,1.601916,True,1
1,0.014417,0.014417,0.0,0,9,0.576562,0.000000,0.000000,70,256,...,0.014417,2.664535e-15,29990391,29.990391,29990391,29.990391,1653023,1.653023,True,2
2,0.010683,0.010683,0.0,0,9,0.496327,0.000000,0.000000,65,256,...,0.010683,0.000000e+00,29987324,29.987324,29987324,29.987324,1601902,1.601902,True,3
3,0.010041,0.010041,0.0,0,7,0.481171,0.000000,0.000000,52,256,...,0.010041,3.330669e-16,29988913,29.988913,29988913,29.988913,1601919,1.601919,True,4
4,0.009150,0.009150,0.0,0,8,0.459324,0.000000,0.000000,62,256,...,0.009150,2.220446e-16,29993947,29.993947,29993947,29.993947,1550304,1.550304,True,5
5,0.020567,0.020567,0.0,0,10,0.688645,0.000000,0.000000,58,256,...,0.020567,1.200429e-15,29988318,29.988318,29988318,29.988318,1601961,1.601961,True,6
6,0.015749,0.015749,0.0,0,8,0.602608,0.000000,0.000000,54,256,...,0.015749,1.776357e-15,29999055,29.999055,29999055,29.999055,1653596,1.653596,True,7
7,0.014379,0.014379,0.0,0,9,0.575801,0.000000,0.000000,64,256,...,0.014379,8.881784e-16,29992571,29.992571,29992571,29.992571,1343600,1.343600,True,8
8,0.013192,0.013192,0.0,0,9,0.551527,0.000000,0.000000,53,256,...,0.013192,9.992007e-16,29996606,29.996606,29996606,29.996606,1601920,1.601920,True,9
9,0.005225,0.005225,0.0,0,8,0.347114,0.000000,0.000000,49,256,...,0.005225,8.604228e-16,29987834,29.987834,29987834,29.987834,1653648,1.653648,True,10


In [11]:
# Safe final close. If interrupted before this cell, rerunning is fine because every event is checkpointed.
try:
    sampler.close()
except Exception:
    pass